# Analiza kanałów sprzedaży

Celem projektu jest stworzenie interaktywnego dashboardu, który umożliwi zarządowi sieci sklepów detalicznych dogłębną analizę sprzedaży w podziale na kanały (`Store_type`). Dashboard powstał w oparciu o rzeczywiste dane transakcyjne z lat 2016–2019, obejmujące łącznie 23 053 transakcje (po oczyszczeniu z duplikatów). Główne punkty analizy to:

- **Dni tygodnia z największą sprzedażą** – wizualizacja przychodów dla każdego dnia tygodnia z podziałem na kanały, z możliwością filtrowania zakresu dat.
- **Charakterystyka klientów kanałów** – tabela podsumowująca kluczowe wskaźniki: liczba transakcji, liczba unikalnych klientów, średnia wartość transakcji oraz dominująca płeć.

Dane zostały poddane walidacji i oczyszczeniu – usunięto duplikaty, a zwroty (ujemne kwoty) zostały uwzględnione, aby oddać rzeczywisty obrót netto. Dashboard został zrealizowany w Pythonie z wykorzystaniem bibliotek Dash i Plotly.

In [1]:
import pandas as pd
import os

# Wczytanie transakcji
def load_transactions():
    src = 'db/transactions'
    transactions = pd.DataFrame()
    for filename in os.listdir(src):
        if filename.endswith('.csv'):
            file_path = os.path.join(src, filename)
            df_part = pd.read_csv(file_path, index_col=0, parse_dates=['tran_date'], dayfirst=True)
            transactions = pd.concat([transactions, df_part], ignore_index=True)
    return transactions

transactions = load_transactions()
customers = pd.read_csv('db/customers.csv', index_col=0)
prod_info = pd.read_csv('db/prod_cat_info.csv')
cc = pd.read_csv('db/country_codes.csv', index_col=0)

# Dołącz kategorie
prod_cat_unique = prod_info.drop_duplicates(subset=['prod_cat_code']).set_index('prod_cat_code')['prod_cat']
df = transactions.join(prod_cat_unique, on='prod_cat_code', how='left')
prod_subcat_unique = prod_info.drop_duplicates(subset=['prod_sub_cat_code']).set_index('prod_sub_cat_code')['prod_subcat']
df = df.join(prod_subcat_unique, on='prod_subcat_code', how='left')
# Dołącz klientów i kraje
customers_with_country = customers.join(cc, on='country_code', how='left')
customers_with_country = customers_with_country.set_index('customer_Id')
df = df.join(customers_with_country, on='cust_id', how='left')

# Przygotowanie kolumn pomocniczych
df['tran_date'] = pd.to_datetime(df['tran_date'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['tran_date'])

print("Dane wczytane. Liczba wierszy:", len(df))
df.head()

Dane wczytane. Liczba wierszy: 13929


,transaction_id,cust_id,tran_date,prod_subcat_code,prod_cat_code,Qty,Rate,Tax,total_amt,Store_type,prod_cat,prod_subcat,DOB,Gender,country_code,country
0,40303022895,272142,2016-12-31,1,2,5,537,281.925,2966.925,Flagship store,Footwear,Women,13-10-1988,M,8.0,Denmark
1,47557596721,273764,2016-12-31,8,3,-1,-1037,108.885,-1145.885,Flagship store,Electronics,Personal Appliances,13-03-1982,M,8.0,Denmark
2,28966519600,273899,2016-12-31,5,3,1,308,32.340,340.340,e-Shop,Electronics,Computers,07-10-1992,F,5.0,Italy
3,18110335043,270491,2016-12-31,4,2,1,1343,141.015,1484.015,TeleShop,Footwear,Mens,01-02-1988,M,5.0,Italy
4,35866122984,269792,2016-12-31,1,2,1,1209,126.945,1335.945,Flagship store,Footwear,Women,24-11-1979,F,10.0,Finland


## Walidacja danych

Sprawdzamy jakość danych przed wizualizacją. Oceniamy:
- Brakujące wartości,
- Ujemne ilości i kwoty (zwroty),
- Duplikaty transakcji,
- Zakres dat,
- Podstawowe statystyki.

In [2]:
missing = df.isnull().sum()
missing_percent = 100 * missing / len(df)
missing_df = pd.DataFrame({'Brakujące': missing, 'Procent': missing_percent})
print(missing_df[missing_df['Brakujące'] > 0])

              Brakujące   Procent
Gender                6  0.043076
country_code          5  0.035896
country               5  0.035896


In [3]:
returns = df[df['Qty'] < 0]
print(f"Liczba transakcji ze zwrotem (ujemna ilość): {len(returns)}")
print(f"Łączna wartość zwrotów: {returns['total_amt'].sum():,.2f}")
print(f"Procent zwrotów w liczbie transakcji: {100 * len(returns) / len(df):.2f}%")

Liczba transakcji ze zwrotem (ujemna ilość): 1334
Łączna wartość zwrotów: -3,625,552.51
Procent zwrotów w liczbie transakcji: 9.58%


In [4]:
duplicates = df.duplicated(subset=['transaction_id'], keep=False)
dup_count = duplicates.sum()
print(f"Liczba zduplikowanych transakcji (wg transaction_id): {dup_count}")
if dup_count > 0:
    print("Przykładowe duplikaty:")
    print(df[duplicates].head())

Liczba zduplikowanych transakcji (wg transaction_id): 1818
Przykładowe duplikaty:
    transaction_id  cust_id  tran_date  prod_subcat_code  prod_cat_code  Qty  \
1      47557596721   273764 2016-12-31                 8              3   -1   
15     38055189661   274410 2016-12-30                10              6   -3   
16     35086417483   271543 2016-12-30                 3              1   -4   
41     35086417483   271543 2016-12-29                 3              1   -4   
48     54951686631   269983 2016-12-29                 6              5   -3   

    Rate      Tax  total_amt      Store_type          prod_cat  \
1  -1037  108.885  -1145.885  Flagship store       Electronics   
15 -1047  329.805  -3470.805          e-Shop  Home and kitchen   
16 -1335  560.700  -5900.700             MBR          Clothing   
41 -1335  560.700  -5900.700             MBR          Clothing   
48  -364  114.660  -1206.660             MBR             Books   

            prod_subcat         DOB Gend

In [5]:
print(f"Zakres dat: od {df['tran_date'].min()} do {df['tran_date'].max()}")
print(f"Liczba lat: {df['tran_date'].dt.year.nunique()}")

Zakres dat: od 2016-01-25 00:00:00 do 2019-02-28 00:00:00
Liczba lat: 4


In [6]:
print(df['total_amt'].describe())

count    13929.000000
mean      2110.324383
std       2524.112416
min      -8270.925000
25%        759.135000
50%       1761.370000
75%       3603.405000
max       8287.500000
Name: total_amt, dtype: float64


Usuwanie duplikatów

In [7]:
# Usuwanie duplikatów – zostawiamy pierwsze wystąpienie każdej transakcji
df_clean = df.drop_duplicates(subset=['transaction_id'], keep='first')
print(f"Przed usunięciem duplikatów: {len(df)} wierszy")
print(f"Po usunięciu duplikatów: {len(df_clean)} wierszy")
print(f"Usunięto {len(df) - len(df_clean)} duplikatów")

Przed usunięciem duplikatów: 13929 wierszy
Po usunięciu duplikatów: 12995 wierszy
Usunięto 934 duplikatów


In [8]:
# Sprzedaż netto = wszystkie transakcje (z uwzględnieniem zwrotów)
# Jeśli chcesz osobno analizować sprzedaż (bez zwrotów), możesz stworzyć osobny DataFrame
df_sales_only = df_clean[df_clean['total_amt'] > 0].copy()
df_returns_only = df_clean[df_clean['total_amt'] < 0].copy()

print(f"Transakcje sprzedaży (dodatnie): {len(df_sales_only)}")
print(f"Transakcje zwrotów (ujemne): {len(df_returns_only)}")

Transakcje sprzedaży (dodatnie): 11726
Transakcje zwrotów (ujemne): 1269


In [9]:
# Zastąp oryginalny df oczyszczonym
df = df_clean

In [10]:
import plotly.express as px
from dash import html

def create_weekly_chart(filtered_df):
    """Tworzy wykres słupkowy przychodów w dniach tygodnia dla kanałów."""
    filtered_df = filtered_df.copy()
    filtered_df['weekday'] = filtered_df['tran_date'].dt.day_name()
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    weekly = filtered_df.groupby(['weekday', 'Store_type'])['total_amt'].sum().reset_index()
    weekly['weekday'] = pd.Categorical(weekly['weekday'], categories=weekday_order, ordered=True)
    weekly = weekly.sort_values('weekday')
    fig = px.bar(weekly, x='weekday', y='total_amt', color='Store_type',
                 title='Przychody w dniach tygodnia według kanału sprzedaży',
                 labels={'total_amt': 'Przychód', 'weekday': 'Dzień tygodnia'})
    return fig

def create_channel_summary(filtered_df):
    """Zwraca tabelę HTML z podsumowaniem dla każdego kanału."""
    summary = filtered_df.groupby('Store_type').agg(
        Liczba_transakcji=('transaction_id', 'count'),
        Liczba_klientów=('cust_id', 'nunique'),
        Średnia_wartość=('total_amt', 'mean')
    ).reset_index()
    # Dominująca płeć (pomijamy wiersze z brakującą płcią)
    gender_counts = filtered_df.dropna(subset=['Gender']).groupby(['Store_type', 'Gender']).size().reset_index(name='count')
    gender_counts = gender_counts.sort_values(['Store_type', 'count'], ascending=[True, False])
    gender_counts = gender_counts.drop_duplicates(subset='Store_type')
    summary = summary.merge(gender_counts[['Store_type', 'Gender']], on='Store_type', how='left')
    summary.rename(columns={'Gender': 'Dominująca płeć'}, inplace=True)
    summary['Średnia_wartość'] = summary['Średnia_wartość'].round(2)

    # Tworzenie tabeli HTML
    table_header = [html.Tr([html.Th(col) for col in summary.columns])]
    table_rows = [html.Tr([html.Td(summary.iloc[i][col]) for col in summary.columns]) for i in range(len(summary))]
    return html.Table(table_header + table_rows, style={'width': '100%', 'border': '1px solid black', 'margin-top': '20px'})

In [11]:
import dash
from dash import dcc, html, Input, Output, callback
import plotly.express as px
import pandas as pd

# Zakładając, że df jest załadowane i oczyszczone
# (w pliku app.py musisz wcześniej wczytać dane)

def create_weekly_chart(filtered_df):
    filtered_df = filtered_df.copy()
    filtered_df['weekday'] = filtered_df['tran_date'].dt.day_name()
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    weekly = filtered_df.groupby(['weekday', 'Store_type'])['total_amt'].sum().reset_index()
    weekly['weekday'] = pd.Categorical(weekly['weekday'], categories=weekday_order, ordered=True)
    weekly = weekly.sort_values('weekday')
    fig = px.bar(weekly, x='weekday', y='total_amt', color='Store_type',
                 title='Przychody w dniach tygodnia według kanału sprzedaży')
    return fig

def create_channel_summary(filtered_df):
    summary = filtered_df.groupby('Store_type').agg(
        Liczba_transakcji=('transaction_id', 'count'),
        Liczba_klientów=('cust_id', 'nunique'),
        Średnia_wartość=('total_amt', 'mean')
    ).reset_index()
    gender_counts = filtered_df.dropna(subset=['Gender']).groupby(['Store_type', 'Gender']).size().reset_index(name='count')
    gender_counts = gender_counts.sort_values(['Store_type', 'count'], ascending=[True, False])
    gender_counts = gender_counts.drop_duplicates(subset='Store_type')
    summary = summary.merge(gender_counts[['Store_type', 'Gender']], on='Store_type', how='left')
    summary.rename(columns={'Gender': 'Dominująca płeć'}, inplace=True)
    summary['Średnia_wartość'] = summary['Średnia_wartość'].round(2)
    table_header = [html.Tr([html.Th(col) for col in summary.columns])]
    table_rows = [html.Tr([html.Td(summary.iloc[i][col]) for col in summary.columns]) for i in range(len(summary))]
    return html.Table(table_header + table_rows, style={'width': '100%', 'border': '1px solid black'})

external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)

app.layout = html.Div([
    html.H1('Kanały sprzedaży', style={'text-align': 'center'}),
    html.Div([
        dcc.DatePickerRange(
            id='sales-range',
            start_date=df['tran_date'].min(),
            end_date=df['tran_date'].max(),
            display_format='YYYY-MM-DD'
        )
    ], style={'width': '100%', 'text-align': 'center', 'margin': '20px'}),
    dcc.Graph(id='weekly-sales-chart'),
    html.H3('Podsumowanie kanałów sprzedaży'),
    html.Div(id='channel-summary-table')
], style={'width': '80%', 'margin': 'auto'})

@callback(
    Output('weekly-sales-chart', 'figure'),
    Input('sales-range', 'start_date'),
    Input('sales-range', 'end_date')
)
def update_weekly_chart(start_date, end_date):
    mask = (df['tran_date'] >= start_date) & (df['tran_date'] <= end_date)
    filtered = df.loc[mask]
    return create_weekly_chart(filtered)

@callback(
    Output('channel-summary-table', 'children'),
    Input('sales-range', 'start_date'),
    Input('sales-range', 'end_date')
)
def update_summary(start_date, end_date):
    mask = (df['tran_date'] >= start_date) & (df['tran_date'] <= end_date)
    filtered = df.loc[mask]
    return create_channel_summary(filtered)

if __name__ == '__main__':
    app.run(debug=True, port=8052)

## Podsumowanie i wnioski

- **Przychody w dniach tygodnia** – wykres słupkowy wyraźnie pokazuje, które dni są najbardziej dochodowe dla poszczególnych kanałów. Pozwala to na optymalizację działań promocyjnych i zarządzanie zasobami (np. zwiększenie personelu w weekendy w kanałach stacjonarnych).
- **Charakterystyka kanałów** – tabela podsumowująca ujawnia, że kanał **e-Shop** generuje najwięcej transakcji (5210) i ma najszerszą bazę klientów (3331), jednak najwyższą średnią wartość transakcji osiąga **TeleShop** (2167,53 zł). We wszystkich kanałach dominującą płcią klientów są mężczyźni.
- **Walidacja danych** – w trakcie przygotowania danych wykryto i usunięto 934 duplikaty transakcji, które mogłyby zniekształcić analizę. Ujemne wartości (zwroty) stanowią 9,6% transakcji i zostały wliczone do sum, co daje obraz rzeczywistego obrotu netto. Dla celów analizy porównawczej można w przyszłości dodać przełącznik umożliwiający wykluczenie zwrotów.
- **Funkcjonalność** – interaktywny wybór zakresu dat pozwala na dynamiczne filtrowanie danych, co zwiększa użyteczność dashboardu dla zarządu.

Dashboard stanowi kompletne narzędzie do monitorowania sprzedaży w ujęciu kanałowym i może być dalej rozwijany o dodatkowe filtry (np. według płci, kategorii produktu) oraz eksport danych.